In [ ]:
from pyspark.sql import functions as F
from pyspark import StorageLevel

from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF, MinHashLSH
from pyspark.ml import Pipeline
from pyspark.ml.functions import vector_to_array
import pandas as pd
import matplotlib.pyplot as plt

df = spark.read.parquet("data/spark-output/cleaned_reviews")


In [ ]:
df2 = (
    df
    .withColumn(
        "ts_sec",
        F.when(F.col("timestamp") > F.lit(10_000_000_000), (F.col("timestamp")/1000).cast("long"))
         .otherwise(F.col("timestamp").cast("long"))
    )
    .withColumn("review_date_fixed", F.to_date(F.from_unixtime(F.col("ts_sec"))))
)

df2.selectExpr(
    "min(review_date_fixed) as min_review_date_fixed",
    "max(review_date_fixed) as max_review_date_fixed",
    "count(*) as n_rows",
    "sum(case when review_date_fixed is null then 1 else 0 end) as n_null_date"
).show(truncate=False)

In [ ]:
chosen_category = "Beauty & Personal Care"

base = (
    df2
    .filter(F.col("main_category_clean") == chosen_category)
    .filter(F.col("review_date_fixed").isNotNull())
    .withColumn(
        "text_all",
        F.lower(
            F.regexp_replace(
                F.concat_ws(" ", F.coalesce(F.col("title"), F.lit("")), F.coalesce(F.col("text"), F.lit(""))),
                "[^a-zA-Z0-9 ]", " "
            )
        )
    )
    .filter(F.length("text_all") > 0)
    .withColumn(
        "rid",
        F.sha2(
            F.concat_ws("||",
                F.coalesce(F.col("parent_asin"), F.lit("")),
                F.coalesce(F.col("asin"), F.lit("")),
                F.coalesce(F.col("user_id"), F.lit("")),
                F.coalesce(F.col("timestamp").cast("string"), F.lit("")),
                F.coalesce(F.col("title"), F.lit("")),
                F.coalesce(F.col("text"), F.lit(""))
            ),
            256
        )
    )
    .select("rid", "parent_asin", "asin", "user_id", "rating", "review_date_fixed", "text_all")
)

base.selectExpr(
    "count(*) as n_rows",
    "min(review_date_fixed) as min_d",
    "max(review_date_fixed) as max_d"
).show(truncate=False)


In [ ]:
old = base.filter(F.col("review_date_fixed") < F.lit("2022-01-01"))
recent = base.filter((F.col("review_date_fixed") >= F.lit("2022-01-01")) & (F.col("review_date_fixed") < F.lit("2024-01-01")))

old.selectExpr("count(*) as n_old", "min(review_date_fixed) as min_old", "max(review_date_fixed) as max_old").show(truncate=False)
recent.selectExpr("count(*) as n_recent", "min(review_date_fixed) as min_recent", "max(review_date_fixed) as max_recent").show(truncate=False)


In [ ]:
N = 80_000

def sample_n(df_in, frac, seed):
    s = (
        df_in
        .select("rid", "text_all")
        .sample(False, frac, seed)
        .limit(N)
        .persist(StorageLevel.MEMORY_AND_DISK)
    )
    _ = s.count()
    return s

old_s = sample_n(old, frac=0.02, seed=42)
recent_s = sample_n(recent, frac=0.04, seed=43)

old_s.count(), recent_s.count()


In [ ]:
tokenizer = Tokenizer(inputCol="text_all", outputCol="words")
remover = StopWordsRemover(inputCol="words", outputCol="filtered")
hashingTF = HashingTF(inputCol="filtered", outputCol="features", numFeatures=1<<18)
pipe = Pipeline(stages=[tokenizer, remover])

def featurize(df_s):
    tmp = pipe.fit(df_s).transform(df_s).filter(F.size("filtered") > 0)
    feat = hashingTF.transform(tmp).filter(F.size(vector_to_array("features")) > 0)
    out = feat.select("rid", "features").persist(StorageLevel.MEMORY_AND_DISK)
    _ = out.count()
    return out

old_feat = featurize(old_s)
recent_feat = featurize(recent_s)

old_feat.count(), recent_feat.count()

In [ ]:
mh = MinHashLSH(inputCol="features", outputCol="hashes", numHashTables=5)

old_model = mh.fit(old_feat)
recent_model = mh.fit(recent_feat)

In [ ]:
def dup_ids(model, df_feat, k):
    h = model.transform(df_feat).select("rid", F.explode("hashes").alias("h"))
    return (
        h.groupBy("h")
         .agg(F.countDistinct("rid").alias("n"))
         .filter(F.col("n") >= k)
         .join(h, on="h", how="inner")
         .select("rid").distinct()
    )

def dup_rate_by_bucket(model, df_feat, k):
    d = dup_ids(model, df_feat, k)
    dup_count = d.count()
    total = df_feat.count()
    return dup_count, total, (dup_count/total if total else 0.0)

metrics = []
for k in [2, 3]:
    o = dup_rate_by_bucket(old_model, old_feat, k)
    r = dup_rate_by_bucket(recent_model, recent_feat, k)
    metrics.append({"period":"pre-2022", "k":k, "dup_count":o[0], "total":o[1], "dup_rate":o[2]})
    metrics.append({"period":"2022-2023", "k":k, "dup_count":r[0], "total":r[1], "dup_rate":r[2]})

metrics_df = pd.DataFrame(metrics)
metrics_df


In [ ]:
plot_df = metrics_df.copy()
print("metrics_df columns:", list(plot_df.columns))

plot_df = plot_df[plot_df["k"] == 3].sort_values("period")

plt.figure(figsize=(6,4))
plt.bar(plot_df["period"], plot_df["dup_rate"])
plt.title("Duplication Rate: Pre-2022 vs 2022–2023 (k=3)")
plt.xlabel("Period")
plt.ylabel("Duplication Rate")
plt.tight_layout()
plt.show()

In [ ]:
k_show = 3

old_dup = dup_ids(old_model, old_feat, k_show)
recent_dup = dup_ids(recent_model, recent_feat, k_show)

print("old dup ids:", old_dup.count(), "recent dup ids:", recent_dup.count())

old_dup.join(old_s, "rid").select("text_all").limit(10).show(truncate=140)
recent_dup.join(recent_s, "rid").select("text_all").limit(10).show(truncate=140)


In [ ]:
base_title = (
    df2
    .filter(F.col("main_category_clean") == chosen_category)
    .filter(F.col("review_date_fixed").isNotNull())
    .withColumn("title_clean", F.lower(F.regexp_replace(F.coalesce(F.col("title"), F.lit("")), "[^a-zA-Z0-9 ]", " ")))
    .filter(F.length("title_clean") > 0)
    .withColumn(
        "rid",
        F.sha2(
            F.concat_ws("||",
                F.coalesce(F.col("parent_asin"), F.lit("")),
                F.coalesce(F.col("asin"), F.lit("")),
                F.coalesce(F.col("user_id"), F.lit("")),
                F.coalesce(F.col("timestamp").cast("string"), F.lit("")),
                F.coalesce(F.col("title"), F.lit("")),
                F.coalesce(F.col("text"), F.lit(""))
            ),
            256
        )
    )
    .select("rid", "review_date_fixed", "title_clean")
)

old_t = base_title.filter(F.col("review_date_fixed") < F.lit("2022-01-01"))
recent_t = base_title.filter((F.col("review_date_fixed") >= F.lit("2022-01-01")) & (F.col("review_date_fixed") < F.lit("2024-01-01")))

def sample_n_title(df_in, frac, seed):
    s = (
        df_in
        .select("rid", F.col("title_clean").alias("text_all"))
        .sample(False, frac, seed)
        .limit(N)
        .persist(StorageLevel.MEMORY_AND_DISK)
    )
    _ = s.count()
    return s

old_t_s = sample_n_title(old_t, frac=0.03, seed=52)
recent_t_s = sample_n_title(recent_t, frac=0.05, seed=53)

old_t_feat = featurize(old_t_s)
recent_t_feat = featurize(recent_t_s)

old_t_model = mh.fit(old_t_feat)
recent_t_model = mh.fit(recent_t_feat)

title_metrics = []
for k in [2, 3]:
    o = dup_rate_by_bucket(old_t_model, old_t_feat, k)
    r = dup_rate_by_bucket(recent_t_model, recent_t_feat, k)
    title_metrics.append({"period":"pre-2022", "k":k, "dup_count":o[0], "total":o[1], "dup_rate":o[2]})
    title_metrics.append({"period":"2022-2023", "k":k, "dup_count":r[0], "total":r[1], "dup_rate":r[2]})

pd.DataFrame(title_metrics)